### Hybrid Retriver

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_astradb import AstraDBVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)



ASTRA_DB_API_ENDPOINT = ""
ASTRA_DB_APPLICATION_TOKEN = ""



vector_store = AstraDBVectorStore(
    collection_name="hybrid_search",
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=None,
)


llm = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434"
)



# dense retriver means normal cosine similary
# Sparse means exact keyword search

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3314.74it/s]


In [96]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]




In [ ]:

# vector_store.add_documents(documents=docs) already added documents
dense_retriever=vector_store.as_retriever()


# Sparse Retriver (BM25)

sparse_retriver = BM25Retriever.from_documents(docs)
sparse_retriver.k=3  ## top k doc to retriver


### Sparse Retriever(BM25)
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ##top- k documents to retriever

##  Combine with Ensemble Retriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weight=[0.7,0.3]
)


In [98]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['AstraDBVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_astradb.vectorstores.AstraDBVectorStore object at 0x0000020367C2BC50>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000020302B2B610>, k=3)], weights=[0.5, 0.5])

In [104]:
query = "How can I build an application using LLMs?"

# Dense results
dense_results = dense_retriever.invoke(query)

print("\n========== DENSE RETRIEVER ==========")

for i, doc in enumerate(dense_results):
    print(f"\n--- Dense Document {i+1} ---")
    print(doc.page_content)


# Sparse results
sparse_results = sparse_retriever.invoke(query)

print("\n========== SPARSE RETRIEVER ==========")

for i, doc in enumerate(sparse_results):
    print(f"\n--- Sparse Document {i+1} ---")
    print(doc.page_content)




========== DENSE RETRIEVER ==========

--- Dense Document 1 ---
LangChain helps build LLM applications.

--- Dense Document 2 ---
Langchain can be used to develop agentic ai application.

--- Dense Document 3 ---
Pinecone is a vector database for semantic search.

--- Dense Document 4 ---
Langchain has many types of retrievers.

========== SPARSE RETRIEVER ==========

--- Sparse Document 1 ---
LangChain helps build LLM applications.

--- Sparse Document 2 ---
Langchain can be used to develop agentic ai application.

--- Sparse Document 3 ---
Langchain has many types of retrievers.


### RAG Pipeline with hybrid retriever

In [100]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question strictly ONLY based on the context below.

Context:
{context}

Question: {input}
""")


In [101]:
### Create stuff Docuemnt Chain
document_chain=create_stuff_documents_chain(llm=llm,prompt=prompt)

## create Full rAg chain
rag_chain=create_retrieval_chain(retriever=hybrid_retriever,combine_docs_chain=document_chain)
rag_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['AstraDBVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_astradb.vectorstores.AstraDBVectorStore object at 0x0000020367C2BC50>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000020302B2B610>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question strictly ONLY based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')


In [102]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)
response
# Step 10: Output
#print("✅ Answer:\n", response["answer"])
 
#print("\n📄 Source Documents:")
#for i, doc in enumerate(response["context"]):
#    print(f"\nDoc {i+1}: {doc.page_content}")

{'input': 'How can I build an app using LLMs?',
 'context': [Document(id='304a1a539d054f5d947cc3df74ea9e0f', metadata={}, page_content='LangChain helps build LLM applications.'),
  Document(id='0bfa7351daa747cc94983bca8eeb1492', metadata={}, page_content='Langchain can be used to develop agentic ai application.'),
  Document(id='88001049d88f43e8a820a02adeb8dbf2', metadata={}, page_content='Langchain has many types of retrievers.'),
  Document(id='6588b29836ba45369895061250d16d2c', metadata={}, page_content='Pinecone is a vector database for semantic search.')],
 'answer': "To build an app using LLMs (Large Language Models), you can leverage platforms and tools like LangChain, which helps in building applications that utilize LLMs. LangChain can be used to develop agentic AI applications, indicating its capability to integrate and utilize LLMs effectively. Additionally, LangChain offers various types of retrievers, which can be utilized to interact with databases or data sources, thereb